# Larry's Sentiment Analysis Project
## Models: Naive Bayes and TextCNN

In this notebook, I am going to build two models to see which one works better for student sentiments. 
1. **Naive Bayes:** This is a simple probabilistic model that uses word counts.
2. **TextCNN:** This is a deep learning model that uses filters (like in image processing) to find patterns in sentences.

I'll follow the standard steps: Loading the data, cleaning it, turning words into numbers, and then training my models.

### Step 1: Import my tools
I'm importing pandas for dataframes, sklearn for Naive Bayes, and Keras/TensorFlow for the CNN.

In [ ]:
import os
import re
import string
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout,
    Concatenate, SpatialDropout1D
)

# Fix random seeds so results are reproducible each run
np.random.seed(42)
tf.random.set_seed(42)

# GloVe paths - check Kaggle input first to avoid download errors
glove_kaggle_path = '/kaggle/input/glove6b100dtxt/glove.6B.100d.txt'
glove_local_path = 'glove.6B.100d.txt'
glove_url = "https://nlp.stanford.edu/data/glove.6B.zip"
glove_zip = "glove.6B.zip"

if os.path.exists(glove_kaggle_path):
    print("Using Kaggle's GloVe dataset.")
    glove_path = glove_kaggle_path
elif not os.path.exists(glove_local_path):
    print("Downloading GloVe vectors from Stanford (this takes a few minutes)...")
    try:
        opener = urllib.request.build_opener()
        opener.addheaders = [('User-agent', 'Mozilla/5.0')]
        urllib.request.install_opener(opener)
        urllib.request.urlretrieve(glove_url, glove_zip)
        print("Download done. Extracting...")
        with zipfile.ZipFile(glove_zip, 'r') as z:
            z.extract(glove_local_path)
        print("GloVe vectors ready.")
        glove_path = glove_local_path
    except Exception as e:
        print(f"Download failed: {e}")
        print("PLEASE ENABLE INTERNET IN KAGGLE SETTINGS OR ADD GLOVE DATASET.")
        glove_path = None
else:
    print("GloVe vectors already available locally.")
    glove_path = glove_local_path

# Load GloVe vectors into memory dictionary
if glove_path:
    print("Loading GloVe vectors into memory...")
    embeddings_lookup = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_lookup[word] = coefs
    print(f"Loaded {len(embeddings_lookup):,} word vectors.")

### Step 2: Load the Data
I am loading the training, validation, and the student test set that has Gmail and WhatsApp messages.

In [ ]:
# Loading from the processed folder
train_df = pd.read_csv('../data/processed/processed_training_dataset.csv')
val_df = pd.read_csv('../data/processed/processed_validation_datset.csv')
test_df = pd.read_csv('../data/processed/student_test_dataset.csv')
print(f"We have {len(train_df)} training samples.")
print(f"We have {len(test_df)} real-world student test samples.")

train_df.head()

## SECTION 1: EDA (Larry - Naive Bayes & TextCNN)

### Step 3: Simple EDA
Let's see how many Positive, Negative, and Neutral messages we have.

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x='sentiment', data=train_df, palette='Set2')
plt.title('How the sentiments are spread in our training data')
plt.show()

print("Counts for each sentiment:")
print(train_df['sentiment'].value_counts())

## SECTION 2: PREPROCESSING

### Step 4: Surgical Cleaning the text
Student communications (especially Gmail and WhatsApp) are full of technical noise like 'MTN:', 'GitHub:', and automated footers. I will use a 'Surgical Cleaning' approach to remove this noise and map student slang to proper English. This helps the model focus on the actual sentiment.

In [ ]:
# --- Slang mapping including WhatsApp shortcuts ---
slang_dictionary = {
    r'\bu\b': 'you', r'\bpls\b': 'please', r'\bplz\b': 'please',
    r'\btmrw\b': 'tomorrow', r'\bwat\b': 'what', r'\bwud\b': 'would',
    r'\bcuz\b': 'because', r'\bbtw\b': 'by the way', r'\bidk\b': 'i do not know',
    r'\bomg\b': 'oh my god', r'\blol\b': 'laughing', r'\bthx\b': 'thanks',
    r'\bsry\b': 'sorry', r'\bwanna\b': 'want to', r'\bgonna\b': 'going to',
    r'\bur\b': 'your', r'\br\b': 'are', r'\bn\b': 'and', r'\bok\b': 'okay'
}

def extract_meta_features(df):
    """
    Extracts structural metadata indicators before cleaning strips punctuation/casing.
    """
    # 1. Count of exclamations and question marks
    df['exclamation_count'] = df['text'].apply(lambda x: str(x).count('!'))
    df['question_count'] = df['text'].apply(lambda x: str(x).count('?'))
    
    # 2. Check for HTML fragments
    df['has_html_artifacts'] = df['text'].apply(lambda x: 1 if re.search(r'<.*?>', str(x)) else 0)
    
    # 3. Check for shout capitalization (ALL CAPS)
    df['is_all_caps'] = df['text'].apply(lambda x: 1 if str(x).isupper() and len(str(x)) > 5 else 0)
    
    # 4. Characters and words length
    df['char_cnt'] = df['text'].apply(lambda x: len(str(x)))
    df['word_cnt'] = df['text'].apply(lambda x: len(str(x).split()))
    
    # 5. Domain Shift Indicators (Technical platform names and billing alerts)
    df['has_platform_mention'] = df['text'].apply(
        lambda x: 1 if re.search(r'\b(github|slack|coursera|udemy|paystack|railway|netlify|heroku)\b', str(x).lower()) else 0
    )
    df['has_service_alert'] = df['text'].apply(
        lambda x: 1 if re.search(r'\b(invoice|billing|terminate|security|alert|reminder)\b', str(x).lower()) else 0
    )
    return df

def surgical_cleaner(text):
    if type(text) != str: return ""
    text = text.lower()
    
    # Remove HTML tags & URLs
    text = re.sub(r'http\S+|www\.\S+|@\w+', '', text)
    text = re.sub(r'<.*?>', '', text)
    
    # Strip common automated signatures
    text = re.sub(r'sent from my (iphone|android|mobile)', '', text)
    text = re.sub(r'please consider the environment before printing', '', text)
    text = re.sub(r'--- forwarded message ---', '', text)
    
    # Standardize slang
    for slang, correct in slang_dictionary.items():
        text = re.sub(slang, correct, text)
        
    # Standardize contractions
    text = re.sub(r"can't", "cannot", text)
    text = re.sub(r"don't", "do not", text)
    text = re.sub(r"isn't", "is not", text)
    
    # Strip punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text if text else "notification"

# --- Process Datasets ---
print("Extracting structural metadata indicators...")
train_df = extract_meta_features(train_df)
val_df   = extract_meta_features(val_df)
test_df  = extract_meta_features(test_df)

print("Applying surgical cleaner...")
train_df['clean'] = train_df['text'].apply(surgical_cleaner)
val_df['clean']   = val_df['text'].apply(surgical_cleaner)
test_df['clean']  = test_df['text'].apply(surgical_cleaner)

print("Surgical cleaning complete!")


### Step 5: Encoding Labels
I will map the sentiments to three clean categories: Negative (0), Neutral (1), and Positive (2). This is important because our test set has a lot of Neutral messages that we don't want to miss!

In [ ]:
# Mapping dictionary for 3-class sentiment
label_mapping = {"Negative": 0, "Neutral": 1, "Positive": 2}

train_df['label'] = train_df['sentiment'].map(label_mapping)
val_df['label'] = val_df['sentiment'].map(label_mapping)
test_df['label'] = test_df['sentiment'].map(label_mapping)

# Drop any rows that couldn't be mapped (like 'Irrelevant' labels)
train_df = train_df.dropna(subset=['label'])
val_df = val_df.dropna(subset=['label'])
test_df = test_df.dropna(subset=['label'])

y_train = train_df['label'].astype(int).values
y_val = val_df['label'].astype(int).values
y_test = test_df['label'].astype(int).values

print("Labels mapped successfully: Negative=0, Neutral=1, Positive=2")

## SECTION 4: TRAINING AND EVALUATION (Larry - Naive Bayes & TextCNN)
### Step 5.1: Tuning Naive Bayes Hyperparameters
I will test different values for 'alpha' (which helps the model handle new words) to find the best setting using our validation set.

# SECTION 3: MODEL TRANSFORMATIONS
### Step 6: Model 1 - Naive Bayes (Classical ML)
I will use the best 'alpha' value I found above to train my final Naive Bayes model.

In [ ]:
# --- Upgraded TF-IDF Vectorizer with Sublinear TF Scaling ---
print("Fitting TF-IDF Vectorizer...")
tfidf_vec = TfidfVectorizer(
    max_features=15000, 
    ngram_range=(1, 3), 
    sublinear_tf=True,  # Logarithmic scaling: 1 + log(tf)
    min_df=2            # Drops noise tokens occurring only once
)

X_train_tfidf = tfidf_vec.fit_transform(train_df['clean'])
X_val_tfidf   = tfidf_vec.transform(val_df['clean'])
X_test_tfidf  = tfidf_vec.transform(test_df['clean'])

print(f"Naive Bayes features shape: {X_train_tfidf.shape}")


In [ ]:
best_alpha = 0.1
best_acc = 0

for a in [0.01, 0.05, 0.1, 0.5, 1.0]:
    temp_nb = MultinomialNB(alpha=a)
    temp_nb.fit(X_train_tfidf, y_train)
    temp_acc = accuracy_score(y_val, temp_nb.predict(X_val_tfidf))
    print(f"Testing NB with alpha={a}, Validation Accuracy: {temp_acc:.4f}")
    
    if temp_acc > best_acc:
        best_acc = temp_acc
        best_alpha = a

print(f"\nBest Alpha: {best_alpha} with Accuracy: {best_acc:.4f}")

In [ ]:
# 2. Initialize and Train Naive Bayes with BEST alpha
nb_clf = MultinomialNB(alpha=best_alpha) 
nb_clf.fit(X_train_tfidf, y_train)

# 3. Predict and evaluate
y_pred_nb = nb_clf.predict(X_val_tfidf)
print("Naive Bayes Validation Accuracy:", round(accuracy_score(y_val, y_pred_nb), 4))
print("\nClassification Report for Naive Bayes:")
print(classification_report(y_val, y_pred_nb, target_names=label_mapping.keys()))

### Step 7: Model 2 - TextCNN (Deep Learning)
For the CNN, we need to tokenize the words into sequences and pad them so they are all the same length.

In [ ]:
MAX_WORDS = 25000
MAX_LEN = 200

# 1. Tokenizing Clean Text
cnn_tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
cnn_tokenizer.fit_on_texts(train_df['clean'])

X_train_seq = pad_sequences(cnn_tokenizer.texts_to_sequences(train_df['clean']), maxlen=MAX_LEN)
X_val_seq   = pad_sequences(cnn_tokenizer.texts_to_sequences(val_df['clean']),   maxlen=MAX_LEN)
X_test_seq  = pad_sequences(cnn_tokenizer.texts_to_sequences(test_df['clean']),  maxlen=MAX_LEN)

# 2. Scale Structural Metadata features
meta_cols = ['exclamation_count', 'question_count', 'has_html_artifacts',
             'is_all_caps', 'char_cnt', 'word_cnt',
             'has_platform_mention', 'has_service_alert']

scaler = StandardScaler()
X_train_meta = scaler.fit_transform(train_df[meta_cols].values)
X_val_meta   = scaler.transform(val_df[meta_cols].values)
X_test_meta  = scaler.transform(test_df[meta_cols].values)

# 3. Build GloVe Embedding Matrix
embedding_matrix = np.zeros((MAX_WORDS, 100))
matched = 0
for word, idx in cnn_tokenizer.word_index.items():
    if idx < MAX_WORDS:
        vec = embeddings_lookup.get(word)
        if vec is not None:
            embedding_matrix[idx] = vec
            matched += 1

print(f"GloVe Coverage for TextCNN: {matched:,} / {MAX_WORDS:,} vocabulary terms matched.")

# --- Build Upgraded Dual-Input TextCNN ---

def build_larry_cnn(learning_rate=1e-3):
    # 1. Text Sequence Branch
    input_layer = Input(shape=(MAX_LEN,), name='text_input')
    embedding = Embedding(
        input_dim=MAX_WORDS,
        output_dim=100,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=True
    )(input_layer)
    s_dropout = SpatialDropout1D(0.3)(embedding)
    
    # Powerful Multi-Filter Convolutional channels
    convs = []
    for fsize in [3, 5, 7, 9]:
        conv = Conv1D(128, fsize, activation='relu')(s_dropout)
        pool = GlobalMaxPooling1D()(conv)
        convs.append(pool)
    
    merged_conv = Concatenate()(convs)
    
    # 2. Metadata Branch
    meta_input = Input(shape=(len(meta_cols),), name='meta_input')
    
    # 3. Concatenate CNN + Metadata
    merged_all = Concatenate()([merged_conv, meta_input])
    
    drop = Dropout(0.5)(merged_all)
    dense = Dense(128, activation='relu')(drop)
    output = Dense(3, activation='softmax')(dense)
    
    model_func = Model(inputs=[input_layer, meta_input], outputs=output)
    model_func.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model_func

larry_cnn = build_larry_cnn()
print(larry_cnn.summary())


### Step 7.1: Tuning TextCNN Parameters
For our deep learning model, we will compare two different learning rates. We want to see which one helps the model perform better on the validation data without overfitting.

In [ ]:
# Calculate class weights for training
classes = np.unique(y_train)
class_weights_arr = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights_dict = dict(zip(classes, class_weights_arr))

# Run quick grid-search to find best learning rate
best_lr_cnn = 0.001
best_val_acc_cnn = 0

for lr in [1e-3, 5e-4]:
    print(f"\n--- Testing TextCNN with Learning Rate: {lr} ---")
    temp_cnn = build_larry_cnn(learning_rate=lr)
    temp_hist = temp_cnn.fit(
        [X_train_seq, X_train_meta], y_train,
        epochs=3,
        batch_size=64,
        validation_data=([X_val_seq, X_val_meta], y_val),
        class_weight=class_weights_dict,
        verbose=1
    )
    val_acc = max(temp_hist.history['val_accuracy'])
    if val_acc > best_val_acc_cnn:
        best_val_acc_cnn = val_acc
        best_lr_cnn = lr

print(f"\nSuccess! Chosen Best Learning Rate: {best_lr_cnn} (Val Accuracy: {best_val_acc_cnn:.4f})")


### Step 8: Training the Final CNN
We train for 15 epochs using our best learning rate found in the experiment above.

In [ ]:
# Train final optimized model for 12 epochs with early stopping
larry_cnn = build_larry_cnn(learning_rate=best_lr_cnn)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = larry_cnn.fit(
    [X_train_seq, X_train_meta], y_train,
    epochs=12,
    batch_size=64,
    validation_data=([X_val_seq, X_val_meta], y_val),
    class_weight=class_weights_dict,
    callbacks=[early_stop],
    verbose=1
)


### Step 9: Testing and Final Comparison
Now the moment of truth! We test both models on the student test dataset (Gmail & WhatsApp) to see if we reached 80% accuracy.

In [ ]:
# Naive Bayes Test Performance
test_pred_nb = nb_clf.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, test_pred_nb)

# TextCNN Test Performance (Requires dual inputs)
test_pred_cnn_probs = larry_cnn.predict([X_test_seq, X_test_meta])
test_pred_cnn = np.argmax(test_pred_cnn_probs, axis=1)
cnn_acc = accuracy_score(y_test, test_pred_cnn)

print(f"Upgraded Naive Bayes Test Accuracy: {nb_acc*100:.2f}%")
print(f"Upgraded TextCNN Test Accuracy: {cnn_acc*100:.2f}%")

# Visual Comparison
plt.figure(figsize=(8, 5))
results = pd.DataFrame({
    'Model': ['Naive Bayes', 'TextCNN'],
    'Accuracy': [nb_acc, cnn_acc]
})
sns.barplot(x='Model', y='Accuracy', data=results, palette='coolwarm', hue='Model', legend=False)
plt.axhline(0.8, color='red', linestyle='--', label='80% Goal')
plt.title('Final Results on Student Communication Data')
plt.ylim(0, 1)
plt.legend()
plt.show()


### Step 10: Emergency Ensemble & Accuracy Boost
If the individual models are still struggling, we can combine them! By averaging the predictions of Naive Bayes and TextCNN, we often get a better result because they catch different types of patterns.

In [ ]:
# 1. Get probabilities from both models
nb_probs = nb_clf.predict_proba(X_test_tfidf)
cnn_probs = larry_cnn.predict([X_test_seq, X_test_meta])  # Updated to dual inputs

# 2. Average the probabilities (Soft Voting)
ensemble_probs = (nb_probs + cnn_probs) / 2
ensemble_preds = np.argmax(ensemble_probs, axis=1)

ensemble_acc = accuracy_score(y_test, ensemble_preds)
print(f"Joint Soft-Voting Ensemble Test Accuracy: {ensemble_acc*100:.2f}%")

print("\n" + "="*50)
print("  OPTIMIZED ENSEMBLE — Cross-Domain Test Report (Gmail + WhatsApp)")
print("="*50)
print(classification_report(y_test, ensemble_preds, target_names=label_mapping.keys()))

# 3. Final Comparison Visualization
plt.figure(figsize=(10, 6))
models_list = ['Naive Bayes', 'TextCNN', 'Ensemble']
accuracies_list = [nb_acc, cnn_acc, ensemble_acc]

sns.barplot(x=models_list, y=accuracies_list, palette='viridis')
plt.axhline(0.8, color='red', linestyle='--', label='80% Target')
plt.title('Final Model Performance Comparison')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.legend()
plt.show()


### Final Recommendations
If we still haven't hit 80%, we should:
1. **Check David's DistilBERT notebook:** Transformers are usually better at handling the complex language in emails.
2. **Check Julianah's Random Forest notebook:** It uses special features like 'exclamation count' which are very helpful for student messages.